In [1]:
import os
import sys
import warnings

os.environ["PYTHONHASHSEED"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
sys.path.append(os.getcwd())
sys.path.append("..")
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import random
import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import models

In [2]:
def fit_ridge_readout(model, x, y, beta=1e-3, batch_size=256):
    
    # build
    _= model(np.empty((1,)+x.shape[1:], np.float32), training=False)
    
    fe = keras.Model(
        inputs=model.inputs,
        outputs=model.layers[-2].output
    )
    
    ds = tf.data.Dataset.from_tensor_slices((x, y)).batch(batch_size)

    D = fe.output_shape[1]
    K = y.shape[1]

    ZTZ = tf.zeros((D, D), dtype=tf.float32)
    YTZ = tf.zeros((K, D), dtype=tf.float32)

    for xb, yb in tqdm.tqdm(ds):
        zb = fe(xb, training=False)          # (B, D)
        ZTZ += tf.matmul(zb, zb, transpose_a=True)      # (D, D)
        YTZ += tf.matmul(yb, zb, transpose_a=True)      # (K, D)

    reg = beta * tf.eye(D, dtype=tf.float32)
    weights = tf.matmul(YTZ, tf.linalg.inv(ZTZ + reg))        # (K, D)
    
    model.layers[-1].set_weights([weights.numpy().T])
    
    return model

In [ ]:
def topk_accuracy(logits, y_true, k=5):
    """
    logits: (N,K)
    y_true: (N,)
    """
    topk = np.argpartition(-logits, kth=k-1, axis=1)[:, :k]
    return np.mean([y_true[i] in topk[i] for i in range(len(y_true))])

def evaluate_model(model, x, y_true, batch_size=256):
    logits = model.predict(x, batch_size=batch_size, verbose=0)  # (N,K)
    y_pred = np.argmax(logits, axis=1)

    acc = accuracy_score(y_true, y_pred)
    top5 = topk_accuracy(logits, y_true, k=5)
    f1m = f1_score(y_true, y_pred, average="macro")

    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, digits=4)

    return {
        "acc": acc,
        "top5": top5,
        "macro_f1": f1m,
        "confusion_matrix": cm,
        "report": report,
    }

In [4]:
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [12]:
# ((x_train, y_train), (x_test, y_test)) = keras.datasets.mnist.load_data()
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = keras.utils.to_categorical(y_train, 10)
y_test = y_test.astype("float32")

H, W, C = x_train.shape[1], x_train.shape[2], x_train.shape[3]
K = y_train.shape[1]

In [13]:
print("x_train:", x_train.shape, x_train.dtype, x_train.min(), x_train.max())
print("y_train:", y_train.shape, y_train.dtype, y_train.min(), y_train.max())
print("x_test:", x_test.shape, x_test.dtype, x_test.min(), x_test.max())
print("y_test:", y_test.shape, y_test.dtype, y_test.min(), y_test.max())

x_train: (50000, 32, 32, 3) float32 0.0 1.0
y_train: (50000, 10) float32 0.0 1.0
x_test: (10000, 32, 32, 3) float32 0.0 1.0
y_test: (10000, 1) float32 0.0 9.0


In [ ]:
model = models.model.get_classifier(
    input_shape=(H, W, C),
    num_classes=K,
    patch_sizes=(4, 4),
    model_type="esn",
    units=256,
    connectivity=0.1,
    leaky=0.9,
    spectral_radius=0.95,
    seed=0,
)

In [8]:
model = fit_ridge_readout(
    model,
    x_train,
    y_train,
    beta=1e-3,
    batch_size=256,
)

100%|██████████| 196/196 [00:40<00:00,  4.89it/s]


In [9]:
metrics = evaluate_model(model, x_test, y_test, batch_size=256)

In [14]:
print("Test Accuracy: {:.4f}".format(metrics["acc"]))
print("Test Top-5 Accuracy: {:.4f}".format(metrics["top5"]))
print("Test Macro F1-score: {:.4f}".format(metrics["macro_f1"]))
print("Classification Report:\n", metrics["report"])

Test Accuracy: 0.1135
Test Top-5 Accuracy: 0.5504
Test Macro F1-score: 0.0613
Classification Report:
               precision    recall  f1-score   support

           0     0.0876    0.4820    0.1483      1000
           1     0.3333    0.0010    0.0020      1000
           2     0.0000    0.0000    0.0000      1000
           3     0.0000    0.0000    0.0000      1000
           4     0.0000    0.0000    0.0000      1000
           5     0.1892    0.0070    0.0135      1000
           6     0.0957    0.3120    0.1464      1000
           7     0.0000    0.0000    0.0000      1000
           8     0.2784    0.3330    0.3033      1000
           9     0.0000    0.0000    0.0000      1000

    accuracy                         0.1135     10000
   macro avg     0.0984    0.1135    0.0613     10000
weighted avg     0.0984    0.1135    0.0613     10000

